In [ ]:
import numpy as np
import h5py
import matplotlib.pyplot as plt

np.random.seed(1)

In [ ]:
def load_data():
    train_dataset = h5py.File('train_catvnoncat.h5', "r")
    train_set_x_orig = np.array(train_dataset["train_set_x"][:])
    train_set_y_orig = np.array(train_dataset["train_set_y"][:])

    test_dataset = h5py.File('test_catvnoncat.h5', "r")
    test_set_x_orig = np.array(test_dataset["test_set_x"][:])
    test_set_y_orig = np.array(test_dataset["test_set_y"][:])

    classes = np.array(test_dataset["list_classes"][:])

    train_set_y_orig = train_set_y_orig.reshape((1, train_set_y_orig.shape[0]))
    test_set_y_orig = test_set_y_orig.reshape((1, test_set_y_orig.shape[0]))

    return train_set_x_orig, train_set_y_orig, test_set_x_orig, test_set_y_orig, classes

train_x_orig, train_y, test_x_orig, test_y, classes = load_data()

print("Train shape:", train_x_orig.shape)
print("Test shape:", test_x_orig.shape)

Train shape: (209, 64, 64, 3)
Test shape: (50, 64, 64, 3)


In [ ]:
train_x = train_x_orig.reshape(train_x_orig.shape[0], -1).T
test_x = test_x_orig.reshape(test_x_orig.shape[0], -1).T

train_x = train_x / 255.
test_x = test_x / 255.

print("Train X:", train_x.shape)
print("Test X:", test_x.shape)

Train X: (12288, 209)
Test X: (12288, 50)


In [ ]:
def sigmoid(Z):
    A = 1/(1+np.exp(-Z))
    cache = Z
    return A, cache

def relu(Z):
    A = np.maximum(0,Z)
    cache = Z
    return A, cache

def sigmoid_backward(dA, cache):
    Z = cache
    s = 1/(1+np.exp(-Z))
    return dA * s * (1-s)

def relu_backward(dA, cache):
    Z = cache
    dZ = np.array(dA, copy=True)
    dZ[Z <= 0] = 0
    return dZ

In [ ]:
def initialize_parameters_deep(layer_dims):
    np.random.seed(3)
    parameters = {}
    L = len(layer_dims)

    for l in range(1, L):
        parameters["W"+str(l)] = np.random.randn(layer_dims[l], layer_dims[l-1]) * np.sqrt(2/layer_dims[l-1])
        parameters["b"+str(l)] = np.zeros((layer_dims[l],1))

    return parameters

In [ ]:
def linear_forward(A, W, b):
    Z = np.dot(W, A) + b
    return Z, (A, W, b)

def linear_activation_forward(A_prev, W, b, activation):
    Z, linear_cache = linear_forward(A_prev, W, b)

    if activation == "relu":
        A, activation_cache = relu(Z)
    else:
        A, activation_cache = sigmoid(Z)

    return A, (linear_cache, activation_cache)

def L_model_forward(X, parameters):
    caches = []
    A = X
    L = len(parameters)//2

    for l in range(1, L):
        A, cache = linear_activation_forward(A,
                                             parameters["W"+str(l)],
                                             parameters["b"+str(l)],
                                             "relu")
        caches.append(cache)

    AL, cache = linear_activation_forward(A,
                                          parameters["W"+str(L)],
                                          parameters["b"+str(L)],
                                          "sigmoid")
    caches.append(cache)

    return AL, caches

In [ ]:
def compute_cost(AL, Y):
    m = Y.shape[1]
    cost = -1/m * np.sum(Y*np.log(AL) + (1-Y)*np.log(1-AL))
    return np.squeeze(cost)

In [ ]:
def linear_backward(dZ, cache):
    A_prev, W, b = cache
    m = A_prev.shape[1]

    dW = 1/m * np.dot(dZ, A_prev.T)
    db = 1/m * np.sum(dZ, axis=1, keepdims=True)
    dA_prev = np.dot(W.T, dZ)

    return dA_prev, dW, db

def linear_activation_backward(dA, cache, activation):
    linear_cache, activation_cache = cache

    if activation == "relu":
        dZ = relu_backward(dA, activation_cache)
    else:
        dZ = sigmoid_backward(dA, activation_cache)

    return linear_backward(dZ, linear_cache)

def L_model_backward(AL, Y, caches):
    grads = {}
    L = len(caches)

    dAL = -(np.divide(Y, AL) - np.divide(1-Y, 1-AL))

    grads["dA"+str(L-1)], grads["dW"+str(L)], grads["db"+str(L)] = \
        linear_activation_backward(dAL, caches[L-1], "sigmoid")

    for l in reversed(range(L-1)):
        dA_prev, dW, db = linear_activation_backward(
            grads["dA"+str(l+1)],
            caches[l],
            "relu"
        )
        grads["dA"+str(l)] = dA_prev
        grads["dW"+str(l+1)] = dW
        grads["db"+str(l+1)] = db

    return grads

In [ ]:
def update_parameters(parameters, grads, learning_rate):
    L = len(parameters)//2

    for l in range(L):
        parameters["W"+str(l+1)] -= learning_rate * grads["dW"+str(l+1)]
        parameters["b"+str(l+1)] -= learning_rate * grads["db"+str(l+1)]

    return parameters

In [ ]:
def L_layer_model(X, Y, layers_dims, learning_rate=0.0075, num_iterations=2500, print_cost=True):

    parameters = initialize_parameters_deep(layers_dims)

    for i in range(num_iterations):

        AL, caches = L_model_forward(X, parameters)
        cost = compute_cost(AL, Y)
        grads = L_model_backward(AL, Y, caches)
        parameters = update_parameters(parameters, grads, learning_rate)

        if print_cost and i % 500 == 0:
            print("Cost after iteration %i: %f" % (i, cost))

    return parameters

In [ ]:
layers_dims = [12288, 50,40,30,20,15,10,8,6,4,1]

parameters = L_layer_model(train_x, train_y, layers_dims)

Cost after iteration 0: 0.748948
Cost after iteration 500: 0.271184
Cost after iteration 1000: 0.003159
Cost after iteration 1500: 0.001040
Cost after iteration 2000: 0.000571


In [ ]:
def predict(X, Y, parameters):
    AL, _ = L_model_forward(X, parameters)
    predictions = (AL > 0.5)

    accuracy = np.mean(predictions == Y)
    print("Accuracy:", accuracy*100, "%")
    return predictions

print("Train:")
predict(train_x, train_y, parameters)

print("Test:")
predict(test_x, test_y, parameters)

Train:
Accuracy: 100.0 %
Test:
Accuracy: 66.0 %


array([[False,  True, False,  True, False,  True, False,  True,  True,
         True,  True,  True,  True,  True, False,  True, False,  True,
        False, False,  True, False, False,  True,  True,  True, False,
        False, False,  True,  True,  True,  True, False, False, False,
        False,  True, False, False, False,  True, False, False,  True,
        False, False,  True, False, False]])

In [ ]:
# 15 Layer Architecture
layers_dims_15 = [12288,
                  100, 90, 80, 70, 60,
                  50, 40, 30, 20,
                  15, 10, 8, 5, 3,
                  1]

parameters_15 = L_layer_model(train_x, train_y, layers_dims_15)

print("15 Layer Model - Train Accuracy:")
predict(train_x, train_y, parameters_15)

print("15 Layer Model - Test Accuracy:")
predict(test_x, test_y, parameters_15)

Cost after iteration 0: 0.707623
Cost after iteration 500: 0.346783
Cost after iteration 1000: 0.001419
Cost after iteration 1500: 0.000331
Cost after iteration 2000: 0.000180
15 Layer Model - Train Accuracy:
Accuracy: 100.0 %
15 Layer Model - Test Accuracy:
Accuracy: 72.0 %


array([[ True,  True, False,  True,  True, False, False,  True,  True,
         True,  True,  True,  True,  True, False,  True, False,  True,
        False, False,  True, False, False,  True,  True,  True, False,
        False, False,  True,  True,  True,  True,  True,  True, False,
        False,  True, False, False, False, False,  True, False,  True,
         True, False,  True,  True, False]])

In [ ]:
# 20 Layer Architecture
layers_dims_20 = [12288,
                  150,140,130,120,110,
                  100,90,80,70,60,
                  50,40,30,25,20,
                  15,10,8,5,
                  1]

parameters_20 = L_layer_model(train_x, train_y, layers_dims_20)

print("20 Layer Model - Train Accuracy:")
predict(train_x, train_y, parameters_20)

print("20 Layer Model - Test Accuracy:")
predict(test_x, test_y, parameters_20)

Cost after iteration 0: 0.684321
Cost after iteration 500: 0.663668
Cost after iteration 1000: 0.181285
Cost after iteration 1500: 0.083901
Cost after iteration 2000: 0.040738
20 Layer Model - Train Accuracy:
Accuracy: 100.0 %
20 Layer Model - Test Accuracy:
Accuracy: 68.0 %


array([[ True, False, False,  True,  True,  True, False,  True,  True,
         True,  True,  True,  True,  True, False,  True, False,  True,
        False, False,  True, False, False, False,  True,  True, False,
        False, False,  True,  True,  True,  True, False, False, False,
        False,  True, False, False, False, False,  True, False,  True,
        False, False,  True,  True, False]])